In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# Φόρτωση δεδομένων
train_path = Path("processed/train_final_scaled.csv")
test_path = Path("processed/test_final_scaled.csv")

assert train_path.exists() and test_path.exists()

features = pd.read_csv(
    train_path, nrows=0
).columns.drop("Label").tolist()

dtypes = {col: "float32" for col in features}
dtypes["Label"] = "int16"

train_df = pd.read_csv(train_path, dtype=dtypes)
test_df = pd.read_csv(test_path, dtype=dtypes)

# 0 = Benign, 1 = Attack
BENIGN_LABEL = 1

y_full = (train_df["Label"].to_numpy() != BENIGN_LABEL).astype(np.float32)
y_test = (test_df["Label"].to_numpy() != BENIGN_LABEL).astype(np.float32)

# Διαχωρισμός Training / Validation (85% / 15%)
train_idx, val_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=0.15,
    random_state=42,
    stratify=y_full
)

X_full = train_df[features].to_numpy(dtype=np.float32)

X_train = X_full[train_idx]
X_val = X_full[val_idx]
X_test = test_df[features].to_numpy(dtype=np.float32)

y_train = y_full[train_idx]
y_val = y_full[val_idx]

# Έλεγχοι
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1] == 44
assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()
assert np.isfinite(X_test).all()

print("=== DATA PREPARATION ===")

for name, X, y in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Testing", X_test, y_test)
]:
    print(
        f"{name}: {X.shape}, "
        f"Benign: {int((y == 0).sum())}, "
        f"Attack: {int((y == 1).sum())}"
    )

print("Features:", len(features))


In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Προετοιμασία δεδομένων
train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train).reshape(-1, 1)
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(y_val).reshape(-1, 1)
)

BATCH_SIZE = 512

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)

# Αρχιτεκτονική MLP
class IntrusionDetectionMLP(nn.Module):
    def __init__(self, input_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


model = IntrusionDetectionMLP(
    input_features=X_train.shape[1]
).to(device)

# Weighted Loss για ανισορροπία κατηγοριών
benign_count = int((y_train == 0).sum())
attack_count = int((y_train == 1).sum())

positive_weight = benign_count / attack_count

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [positive_weight],
        dtype=torch.float32,
        device=device
    )
)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.001
)

print("=== MLP MODEL ===")
print(model)
print("Device:", device)
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
print("Positive class weight:", round(positive_weight, 4))


In [ ]:

import time
from sklearn.metrics import precision_score, recall_score, f1_score

EPOCHS = 15
PATIENCE = 4

Path("models").mkdir(exist_ok=True)
model_path = Path("models/mlp_binary_best.pt")

history = []
best_f1 = -1
patience_counter = 0

for epoch in range(EPOCHS):
    start = time.perf_counter()

    # Training
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    train_loss = total_loss / len(train_dataset)

    # Validation
    model.eval()
    predictions, targets = [], []

    with torch.inference_mode():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch.to(device))

            predictions.extend(
                (logits >= 0).int().cpu().numpy().ravel()
            )
            targets.extend(y_batch.numpy().ravel())

    precision = precision_score(targets, predictions, zero_division=0)
    recall = recall_score(targets, predictions, zero_division=0)
    f1 = f1_score(targets, predictions, zero_division=0)

    epoch_time = time.perf_counter() - start

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_precision": precision,
        "val_recall": recall,
        "val_f1": f1,
        "epoch_time": epoch_time
    })

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f} | "
        f"Time: {epoch_time:.1f}s",
        flush=True
    )

    # Early Stopping
    if f1 > best_f1:
        best_f1 = f1
        patience_counter = 0
        torch.save(model.state_dict(), model_path)
    else:
        patience_counter += 1

        if patience_counter >= PATIENCE:
            print("\nEarly stopping triggered!")
            break

# Φόρτωση καλύτερου μοντέλου
model.load_state_dict(
    torch.load(model_path, map_location=device, weights_only=True)
)
model.eval()

print("\n=== TRAINING COMPLETED ===")
print("Epochs:", len(history))
print("Best Validation F1:", round(best_f1, 4))
print("Saved model:", model_path)


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

Path("results").mkdir(exist_ok=True)

# Προβλέψεις στο Testing Set
model.eval()
test_preds = []

with torch.inference_mode():
    for i in range(0, len(X_test), 512):
        batch = torch.from_numpy(X_test[i:i+512]).to(device)
        logits = model(batch)
        test_preds.append((logits >= 0).int().cpu().numpy().ravel())

y_pred = np.concatenate(test_preds)
y_true = y_test.astype(int)

# Υπολογισμός μετρικών
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

metrics = {
    "Model": "MLP",
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision": precision_score(y_true, y_pred),
    "Recall": recall_score(y_true, y_pred),
    "F1": f1_score(y_true, y_pred),
    "FPR": fp / (fp + tn),
    "Specificity": tn / (tn + fp),
    "TN": tn, "FP": fp, "FN": fn, "TP": tp
}

print("=== MLP TEST RESULTS ===")
for name in ["Accuracy", "Precision", "Recall", "F1", "FPR"]:
    print(f"{name}: {metrics[name]:.6f}")

print("\nConfusion Matrix:")
print(cm)

# Αποθήκευση αποτελεσμάτων
pd.DataFrame([metrics]).to_csv(
    "results/mlp_test_metrics.csv", index=False
)

# Confusion Matrix Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Benign", "Attack"],
    yticklabels=["Benign", "Attack"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("MLP - Confusion Matrix")
plt.tight_layout()
plt.savefig("results/mlp_confusion_matrix.png", dpi=150)
plt.show()


In [ ]:

import json

model.eval()
X_tensor = torch.from_numpy(
    np.ascontiguousarray(X_test)
).to(device)

BATCH_SIZE_TEST = 512

# Warm-up
with torch.inference_mode():
    for _ in range(100):
        model(X_tensor[:1])

    for _ in range(20):
        model(X_tensor[:BATCH_SIZE_TEST])

# 1. Single-flow latency (2000 μετρήσεις)
single_times = []

with torch.inference_mode():
    for i in range(2000):
        start = time.perf_counter_ns()
        model(X_tensor[i:i+1])
        single_times.append(
            (time.perf_counter_ns() - start) / 1_000_000
        )

mean_latency = float(np.mean(single_times))
median_latency = float(np.median(single_times))
p95_latency = float(np.percentile(single_times, 95))

# 2. Batch throughput (5 επαναλήψεις)
batch_times = []

with torch.inference_mode():
    for _ in range(5):
        start = time.perf_counter()

        for i in range(0, len(X_tensor), BATCH_SIZE_TEST):
            model(X_tensor[i:i+BATCH_SIZE_TEST])

        batch_times.append(time.perf_counter() - start)

total_time = float(np.median(batch_times))
throughput = len(X_tensor) / total_time
amortized_ms = total_time / len(X_tensor) * 1000

print("=== INFERENCE RESULTS ===")
print(f"Mean latency:   {mean_latency:.6f} ms")
print(f"Median latency: {median_latency:.6f} ms")
print(f"P95 latency:    {p95_latency:.6f} ms")
print(f"Batch throughput: {throughput:.2f} flows/s")
print(f"Total test time:  {total_time:.4f} s")

# Αποθήκευση για την τελική σύγκριση
results = {
    "model": "MLP",
    "device": str(device),
    "torch_version": torch.__version__,
    "cpu_threads": torch.get_num_threads(),
    "test_samples": len(X_tensor),
    "single_flow_mean_ms": mean_latency,
    "single_flow_median_ms": median_latency,
    "single_flow_p95_ms": p95_latency,
    "batch_size": BATCH_SIZE_TEST,
    "batch_total_median_seconds": total_time,
    "throughput_flows_per_second": throughput,
    "batch_amortized_ms_per_flow": amortized_ms
}

with open("results/mlp_inference_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nBenchmark completed!")


In [ ]:

# Φόρτωση αποτελεσμάτων ταξινόμησης
classification = pd.read_csv(
    "results/mlp_test_metrics.csv"
).iloc[0]

# Φόρτωση αποτελεσμάτων inference
with open("results/mlp_inference_results.json") as f:
    inference = json.load(f)

# Μέγεθος μοντέλου
model_size = Path("models/mlp_binary_best.pt").stat().st_size / 1024

# Συγκεντρωτικός πίνακας
summary = pd.DataFrame([{
    "Model": "MLP",
    "Accuracy": classification["Accuracy"],
    "Precision": classification["Precision"],
    "Recall": classification["Recall"],
    "F1": classification["F1"],
    "FPR": classification["FPR"],
    "Mean Latency (ms)": inference["single_flow_mean_ms"],
    "Median Latency (ms)": inference["single_flow_median_ms"],
    "P95 Latency (ms)": inference["single_flow_p95_ms"],
    "Throughput (flows/s)": inference["throughput_flows_per_second"],
    "Model Size (KB)": model_size,
    "Device": inference["device"]
}])

summary.to_csv(
    "results/mlp_final_summary.csv",
    index=False
)

print("=== MLP FINAL SUMMARY ===")
print(summary.T.to_string(header=False))
print("\nResults saved successfully!")
